%md
### 🔗 Links and Resources
| Method | Description |
|--------|-------------|
| [DataFrame.union()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.union.html) | Combines rows of two DataFrames with the same schema (by column position) |
| [DataFrame.unionAll()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unionAll.html) | Alias for `union()` — kept for backward compatibility |
| [DataFrame.unionByName()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unionByName.html) | Combines rows by matching column names (not position); supports `allowMissingColumns=True` |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

📌 Creating DataFrames

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_sales = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("sales_amount", IntegerType(), True)
])

# Q1 sales data
data_q1 = [
    (1001, 101, "2025-01-15", 5000),
    (1002, 102, "2025-01-16", 7000),
    (1003, 103, "2025-02-10", 6500),
]

# Q2 sales data
data_q2 = [
    (1004, 101, "2025-04-05", 4800),
    (1005, 104, "2025-05-12", 5300),
    (1006, 102, "2025-06-20", 9100),
]

df_q1 = spark.createDataFrame(data_q1, schema=schema_sales)
df_q2 = spark.createDataFrame(data_q2, schema=schema_sales)

print("Q1 Sales:")
df_q1.show()

print("Q2 Sales:")
df_q2.show()

📌 union()

In [ ]:
# union() combines rows from two DataFrames by column position.
# Both DataFrames must have the same number of columns and compatible types.
# Duplicate rows are NOT removed — use .distinct() to deduplicate.
df_union = df_q1.union(df_q2)

df_union.show()

📌 unionAll()

In [ ]:
# unionAll() is an alias for union() — behaves identically.
# Kept for compatibility with older Spark/SQL conventions.
df_union_all = df_q1.unionAll(df_q2)

df_union_all.show()

📌 unionByName()

In [ ]:
# unionByName() matches columns by name instead of position.
# Useful when two DataFrames have the same columns but in different order.

# Reorder columns in df_q2 to demonstrate position-independence
df_q2_reordered = df_q2.select("sales_amount", "sale_date", "store_id", "transaction_id")

# union() would produce wrong results here (columns misaligned by position)
# unionByName() resolves correctly by matching on column names
df_union_by_name = df_q1.unionByName(df_q2_reordered)

df_union_by_name.show()

📌 unionByName() with allowMissingColumns

In [ ]:
# allowMissingColumns=True fills NULL for columns that exist in one DataFrame but not the other.
# Useful when unioning DataFrames with different (but overlapping) schemas.

schema_online = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("sales_amount", IntegerType(), True),
    StructField("channel", StringType(), True),   # extra column not in df_q1
])

data_online = [
    (2001, 105, "2025-03-01", 3200, "web"),
    (2002, 106, "2025-03-15", 4100, "mobile"),
]

df_online = spark.createDataFrame(data_online, schema=schema_online)

# df_q1 has no 'channel' column — without allowMissingColumns this would fail
df_combined = df_q1.unionByName(df_online, allowMissingColumns=True)

df_combined.show()

---
## 🏪 Store DataFrames Examples

The following examples use two store DataFrames with **different schemas**:
- `df_1` — 4 columns: `id`, `store_name`, `city`, `country`
- `df_2` — 3 columns: `id`, `store_name`, `city` (no `country`)

This highlights key Union behaviours when schemas don't match.

📌 Creating Store DataFrames

In [ ]:
# df_1 has 4 columns including 'country'
schema_stores_1 = StructType([
    StructField("id", IntegerType(), True),
    StructField("store_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True)
])

data_stores_1 = [
    (101, "Store A", "New York", "USA"),
    (102, "Store B", "Los Angeles", "USA"),
    (103, "Store C", "Chicago", "USA"),
    (104, "Store D", "Houston", "USA"),
    (105, "Store E", "Phoenix", "USA")
]

df_1 = spark.createDataFrame(data_stores_1, schema_stores_1)

# df_2 has 3 columns — no 'country'
schema_stores_2 = StructType([
    StructField("id", IntegerType(), True),
    StructField("store_name", StringType(), True),
    StructField("city", StringType(), True)
])

data_stores_2 = [
    (101, "Store A", "New York"),
    (102, "Store B", "Los Angeles"),
    (103, "Store C", "Chicago"),
    (104, "Store D", "Houston"),
    (105, "Store E", "Phoenix")
]

df_2 = spark.createDataFrame(data_stores_2, schema_stores_2)

print("df_1 schema:", df_1.schema.simpleString())
df_1.show()

print("df_2 schema:", df_2.schema.simpleString())
df_2.show()

📌 union() fails when column counts differ

In [ ]:
# union() aligns by column position and requires an equal number of columns.
# df_1 has 4 columns, df_2 has 3 — this will raise an AnalysisException.

try:
    df_1.union(df_2).show()
except Exception as e:
    print(f"Error: {e}")

📌 union() after adding the missing column to df_2

In [ ]:
from pyspark.sql.functions import lit

# Add a NULL 'country' column to df_2 so its schema matches df_1.
# union() then works because both DataFrames have 4 columns in the same order.
df_2_with_country = df_2.withColumn("country", lit(None).cast(StringType()))

df_union_stores = df_1.union(df_2_with_country)

print("union() result (df_1 ∪ df_2 with NULL country):")
df_union_stores.show()

📌 unionByName() with allowMissingColumns — the cleanest solution

In [ ]:
# unionByName() matches columns by name rather than position.
# allowMissingColumns=True fills NULL for columns absent in either DataFrame.
# No manual schema alignment needed — Spark handles it automatically.
df_union_by_name_stores = df_1.unionByName(df_2, allowMissingColumns=True)

print("unionByName(allowMissingColumns=True) result:")
df_union_by_name_stores.show()

📌 Deduplication with .distinct() after union

In [ ]:
# union() keeps all rows including duplicates.
# Chain .distinct() to remove exact row duplicates from the combined result.
# Here df_1 and df_2 share the same store ids/names/cities,
# so after aligning schemas the overlapping rows will be deduplicated.

df_distinct_stores = df_union_by_name_stores.distinct()

print(f"Before distinct: {df_union_by_name_stores.count()} rows")
print(f"After  distinct: {df_distinct_stores.count()} rows")
print()
df_distinct_stores.orderBy("id").show()